# Notebook 01 — Data Collection and Preprocessing

Downloads and preprocesses three real-world email datasets from Kaggle.

## What This Notebook Does
- Configures Kaggle API credentials
- Downloads Enron, Nazario, and supplementary phishing datasets
- Extracts email body text from raw Enron message headers
- Cleans and normalises all email text
- Balances dataset to equal class sizes
- Saves processed dataset to disk

## Inputs
- Kaggle API credentials (~/.kaggle/kaggle.json)

## Outputs
- data/processed/dataset_phase1.csv — 4,000 emails (2 classes)
- data/processed/dataset_final.csv — 10,492 emails (3 classes)
- data/processed/dataset_cleaned.csv — cleaned AI phishing emails
- data/processed/dataset_diverse.csv — 12,582 emails (diverse sources)

## Runtime
Approximately 2-3 minutes (excluding dataset downloads)

Follow the instructions in data/README.md to download them before running this notebook.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json
from pathlib import Path

# Set up project paths
BASE_DIR = Path("C:/phishing_detection")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_GENERATED = BASE_DIR / "data" / "generated"

# Create folders if they don't exist
for folder in [DATA_RAW, DATA_PROCESSED, DATA_GENERATED]:
    folder.mkdir(parents=True, exist_ok=True)

print("All folders ready:")
print(f"  Raw data:       {DATA_RAW}")
print(f"  Processed data: {DATA_PROCESSED}")
print(f"  Generated data: {DATA_GENERATED}")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "kaggle"], capture_output=True)
print("Kaggle installed")

In [ ]:
import os
import json
from pathlib import Path

kaggle_creds = {
    "username": "xyz",
    "key": "key"
}

# Save credentials to the right location
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / "kaggle.json"

with open(creds_path, "w") as f:
    json.dump(kaggle_creds, f)

# Set correct permissions
import stat
os.chmod(creds_path, stat.S_IRUSR | stat.S_IWUSR)

print("Kaggle credentials saved to:", creds_path)

In [ ]:
import subprocess

datasets = [
    ("wcukierski/enron-email-dataset", "enron"),
    ("subhajournal/phishingemails", "nazario"),
    ("ethancratchley/phishing-emails", "ai_phishing"),
]

for dataset_id, folder_name in datasets:
    save_path = DATA_RAW / folder_name
    save_path.mkdir(exist_ok=True)
    print(f"Downloading {dataset_id}")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset_id,
         "--unzip", "-p", str(save_path)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  Done: {folder_name}")
    else:
        print(f"  Error: {result.stderr}")

print("\nAll downloads complete.")

In [ ]:
alternatives = [
    "charlottehase/phishing-email-detection-dataset",
    "naserabdullahalam/phishing-email-dataset",
    "hassanamin/email-spam-dataset",
]

for dataset_id in alternatives:
    print(f"Trying {dataset_id}...")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset_id,
         "--unzip", "-p", str(DATA_RAW / "ai_phishing")],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  SUCCESS: {dataset_id}")
        break
    else:
        print(f"  Failed, trying next")

print("Done.")

In [ ]:
for folder in ["enron", "nazario", "ai_phishing"]:
    path = DATA_RAW / folder
    files = list(path.glob("*"))
    print(f"\n{folder}/")
    for f in files:
        size_mb = f.stat().st_size / (1024*1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")

In [ ]:
import pandas as pd

# Enron
enron_df = pd.read_csv(DATA_RAW / "enron" / "emails.csv")
print(" ENRON ")
print("Shape:", enron_df.shape)
print("Columns:", enron_df.columns.tolist())
print(enron_df.head(2))

In [ ]:
# Nazario
nazario_df = pd.read_csv(DATA_RAW / "nazario" / "Phishing_Email.csv")
print(" NAZARIO ")
print("Shape:", nazario_df.shape)
print("Columns:", nazario_df.columns.tolist())
print(nazario_df.head(2))

In [ ]:
# AI Phishing - auto find the CSV
ai_files = list((DATA_RAW / "ai_phishing").glob("*.csv"))
print("Files found:", ai_files)

ai_df = pd.read_csv(ai_files[0])
print("\nAI PHISHING ")
print("Shape:", ai_df.shape)
print("Columns:", ai_df.columns.tolist())
print(ai_df.head(2))

In [ ]:
phishing_df = pd.read_csv(DATA_RAW / "ai_phishing" / "phishing_email.csv")
print(" PHISHING_EMAIL.CSV ")
print("Shape:", phishing_df.shape)
print("Columns:", phishing_df.columns.tolist())
print("\nFirst 2 rows:")
print(phishing_df.head(2))
print("\nUnique labels:", phishing_df.iloc[:, -1].unique() if phishing_df.shape[1] > 1 else "N/A")

In [ ]:
print("Nazario label values:")
print(nazario_df["Email Type"].value_counts())

In [ ]:
#load and clean Enron (legitimate emails)
import re

def extract_email_body(raw_message):
    """Extract just the body text from a raw Enron email string."""
    # Split on blank line which separates headers from body
    parts = re.split(r'\n\n', raw_message, maxsplit=1)
    if len(parts) == 2:
        body = parts[1].strip()
    else:
        body = raw_message.strip()
    # Remove forwarded/replied headers
    body = re.sub(r'-{3,}.*?-{3,}', '', body, flags=re.DOTALL)
    # Remove lines starting with > (quoted replies)
    body = '\n'.join([l for l in body.split('\n') if not l.startswith('>')])
    # Collapse whitespace
    body = re.sub(r'\s+', ' ', body).strip()
    return body

print("Loading Enron dataset ")
enron_raw = pd.read_csv(DATA_RAW / "enron" / "emails.csv")

print("Extracting email bodies")
enron_raw['text'] = enron_raw['message'].apply(extract_email_body)

# Keep only emails with meaningful content (more than 50 characters)
enron_clean = enron_raw[enron_raw['text'].str.len() > 50][['text']].copy()
enron_clean['label'] = 0  # 0 = legitimate

# Sample 3000 to keep dataset balanced
enron_sample = enron_clean.sample(n=3000, random_state=42).reset_index(drop=True)

print(f"Enron cleaned: {len(enron_sample)} emails")
print("Sample:")
print(enron_sample['text'].iloc[0][:200])

In [ ]:
#load and clean Nazario (legitimate and human phishing emails)
nazario_raw = pd.read_csv(DATA_RAW / "nazario" / "Phishing_Email.csv")

# Rename columns
nazario_raw = nazario_raw.rename(columns={"Email Text": "text", "Email Type": "email_type"})

# Map labels: Safe Email = 0 (legitimate), Phishing Email = 1 (human phishing)
nazario_raw['label'] = nazario_raw['email_type'].map({
    'Safe Email': 0,
    'Phishing Email': 1
})

# Drop rows with missing text or label
nazario_clean = nazario_raw[['text', 'label']].dropna()
nazario_clean = nazario_clean[nazario_clean['text'].str.len() > 50]

# Sample to balance: 2000 legitimate, 3000 phishing
nazario_legit = nazario_clean[nazario_clean['label'] == 0].sample(n=2000, random_state=42)
nazario_phish = nazario_clean[nazario_clean['label'] == 1].sample(n=3000, random_state=42)
nazario_sample = pd.concat([nazario_legit, nazario_phish]).reset_index(drop=True)

print(f"Nazario cleaned: {len(nazario_sample)} emails")
print("Label distribution:")
print(nazario_sample['label'].value_counts())

In [ ]:
#load and clean phishing email dataset
phishing_raw = pd.read_csv(DATA_RAW / "ai_phishing" / "phishing_email.csv")

# Rename columns
phishing_raw = phishing_raw.rename(columns={"text_combined": "text"})

# Drop missing
phishing_clean = phishing_raw[['text', 'label']].dropna()
phishing_clean = phishing_clean[phishing_clean['text'].str.len() > 50]

# label 0 = legitimate, label 1 = human phishing
# Sample 1000 of each to supplement
phishing_legit = phishing_clean[phishing_clean['label'] == 0].sample(n=1000, random_state=42)
phishing_phish = phishing_clean[phishing_clean['label'] == 1].sample(n=1000, random_state=42)
phishing_sample = pd.concat([phishing_legit, phishing_phish]).reset_index(drop=True)

print(f"Phishing dataset cleaned: {len(phishing_sample)} emails")
print("Label distribution:")
print(phishing_sample['label'].value_counts())

In [ ]:
# Combine all sources
combined = pd.concat([
    enron_sample,
    nazario_sample,
    phishing_sample
], ignore_index=True)

# Final cleaning
combined['text'] = combined['text'].astype(str).str.strip()
combined = combined[combined['text'].str.len() > 50]
combined = combined.dropna(subset=['label'])
combined['label'] = combined['label'].astype(int)

# Shuffle
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

print("=== FINAL DATASET ===")
print(f"Total emails: {len(combined)}")
print("\nClass distribution:")
print(combined['label'].value_counts().sort_index())
print("\nLabel meanings:")
print("  0 = Legitimate")
print("  1 = Human phishing")
print("  2 = AI-generated phishing (added in Week 3)")
print("\nSample row:")
print(combined.iloc[0])

In [ ]:
save_path = DATA_PROCESSED / "dataset_phase1.csv"
combined.to_csv(save_path, index=False)
print(f"Dataset saved to: {save_path}")
print(f"File size: {save_path.stat().st_size / (1024*1024):.1f} MB")

In [ ]:

# Just run Cell 1 (imports) then run this cell

dataset = pd.read_csv(DATA_PROCESSED / "dataset_phase1.csv")
print(f"Dataset loaded: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load existing dataset
df = pd.read_csv(DATA_PROCESSED / "dataset_phase1.csv")
print(f"Loaded: {len(df)} emails")
print(df['label'].value_counts().sort_index())

# Trim to 2,000 per class temporarily (Week 3 will add AI phishing to make 4,000 per class)
legit = df[df['label'] == 0].sample(n=2000, random_state=42)
human_phish = df[df['label'] == 1].sample(n=2000, random_state=42)

df_trimmed = pd.concat([legit, human_phish]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nTrimmed dataset:")
print(df_trimmed['label'].value_counts().sort_index())

# Save
df_trimmed.to_csv(DATA_PROCESSED / "dataset_phase1.csv", index=False)
print("\nSaved successfully.")

In [ ]:
#run this when you open
import pandas as pd
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "dataset_phase1.csv")
print(f"Dataset loaded: {len(df)} emails")
print(df['label'].value_counts().sort_index())